# Explicação do `run_full_factorial.sh`

Para automatizar o processo de geração de dados foi criado o script `run_full_factorial.sh`, que realiza estas funções:

1. Compilação de gerador.cpp e sorting.cpp;
2. Teste full factorial;
3. 5 repetições do teste;

Todo script foi gerado utilizando IA do ChatGPT para codificá-lo.

# Estrutura esperada do projeto

```text
src/
│
├── generador.cpp
├── sorting.cpp
├── run_full_factorial.sh
├── datasets/
└── results/
```
Caso o run_full_factorial.sh não esteja na pasta com os arquivos .cpp, não funcionará

# Código completo do `run_full_factorial.sh`

In [ ]:
#!/usr/bin/env bash

set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
cd "$SCRIPT_DIR"

GERADOR_EXE="$SCRIPT_DIR/gerador"
SORTING_EXE="$SCRIPT_DIR/sorting"
DATASETS_DIR="$SCRIPT_DIR/datasets"
RESULTS_DIR="$SCRIPT_DIR/results"

mkdir -p "$DATASETS_DIR"
mkdir -p "$RESULTS_DIR"

echo "Compilando gerador..."
g++ gerador.cpp -std=c++17 -O2 -o "$GERADOR_EXE"

echo "Compilando sorting..."
g++ sorting.cpp -std=c++17 -O2 -pthread -o "$SORTING_EXE"

ALGORITHMS=(heap merge quick)
STRESSES=(none cpu ram both)
SIZES=(8)
REPS=(1 2 3 4 5)

# evita interferência do scheduler (melhor esforço)
export OMP_NUM_THREADS=1

run_benchmark () {
  local rep=$1
  local size=$2
  local dataset=$3
  local algorithm=$4
  local stress=$5

  echo "RUN | Rep $rep | Size 10^$size | $(basename "$dataset") | $algorithm | $stress"

  # pequeno warm-up (reduz cold-start effects)
  "$SORTING_EXE" "$algorithm" "$dataset" "$stress" >/dev/null 2>&1 || true

  # execução real
  "$SORTING_EXE" "$algorithm" "$dataset" "$stress"
}

for rep in "${REPS[@]}"; do
  echo "=== Repetição $rep / ${#REPS[@]} ==="

  for size in "${SIZES[@]}"; do
    echo "Gerando datasets para tamanho 10^$size..."
    "$GERADOR_EXE" "$size" "$rep"

    mapfile -t dataset_files < <(
      find "$DATASETS_DIR" -maxdepth 1 -type f -name '*.bin' | sort
    )

    # 🔀 randomiza ordem dos datasets (reduz viés temporal)
    mapfile -t dataset_files < <(
      printf "%s\n" "${dataset_files[@]}" | shuf
    )

    for dataset in "${dataset_files[@]}"; do
      algorithms_shuffled=($(printf "%s\n" "${ALGORITHMS[@]}" | shuf))
      stresses_shuffled=($(printf "%s\n" "${STRESSES[@]}" | shuf))

      for algorithm in "${algorithms_shuffled[@]}"; do
        for stress in "${stresses_shuffled[@]}"; do

          run_benchmark "$rep" "$size" "$dataset" "$algorithm" "$stress"

          # 💤 pausa leve para reduzir efeito de turbo boost contínuo / thermal throttling
          sleep 0.2

        done
      done
    done

    # 💤 pausa maior entre tamanhos (importante para “cooldown térmico”)
    echo "Cooldown entre tamanhos..."
    sleep 2

  done

done

echo "Full factorial concluído. Resultados em: $RESULTS_DIR"


# Explicação do Script

O script utiliza uma seed base (SEED = 10) para garantir a reprodutibilidade dos experimentos, assegurando que os mesmos vetores sejam gerados em execuções futuras.

O funcionamento consiste em um processo automatizado de benchmark. Para cada repetição, o script executa o gerador de datasets, produzindo arquivos .bin dentro da pasta datasets/.

Em seguida, o script percorre todos os arquivos dessa pasta e executa os três algoritmos de ordenação (heap, merge e quick), combinando cada um deles com diferentes configurações de estresse de sistema. Cada execução utiliza um dataset específico como entrada.

Os resultados de cada execução são registrados em um arquivo results.csv dentro da pasta results/, permitindo posterior análise estatística e comparação de desempenho entre algoritmos e cenários.

Durante a execução, o script exibe mensagens de progresso, informando o estado atual do experimento (repetição, tamanho, dataset, algoritmo e configuração de estresse). Isso facilita o acompanhamento do processo e ajuda a identificar possíveis gargalos ou etapas ainda pendentes.

# Tamanho da entrada

Foi escolhido que todos vetores terão tamanho de 10^8 (100 milhões de instâncias), pois o grupo considerou que é o ponto ideal entre vetores longos suficientes para que o tempo de execução seja menos afetado por fatores randômicos e incontroláveis do hardware e vetores curtos suficientes para o tempo de benchmark ser acessível


# Execuções
8 vetores * 3 algoritmos * 4 níveis de estresse = 96 tipos

96 tipos * 5 repetições = 480 execuções

# Estrutura Experimental

O benchmark segue uma execução em full factorial com repetição, combinando diferentes tamanhos de entrada, datasets, algoritmos de ordenação e cenários de estresse do sistema.

A estrutura geral pode ser descrita como:

```text
para cada repetição:
    gerar datasets (com seed definida)
    para cada dataset gerado:
        para cada algoritmo (heap, merge, quick):
            para cada cenário de stress (none, cpu, ram, both):
                executar benchmark
                salvar resultado
```

Cada execução do benchmark é realizada como um processo separado, o que permite:

isolamento de memória entre execuções;
medição mais precisa do pico de uso de RAM;
redução de interferência entre algoritmos e cenários de stress;
maior estabilidade na coleta de métricas de desempenho e energia.

Além disso, o script inclui pausas curtas entre execuções e pausas maiores entre tamanhos de entrada, reduzindo efeitos de aquecimento térmico e variações causadas pelo sistema operacional.

# Execução

## Dar permissão

```bash
chmod +x run_full_factorial.sh
```

## Executar

```bash
./run_full_factorial.sh
```